# Pretraining — Foundations
> Section 01 — Topic 05 — foundations

**Prerequisites:** [04-transformer-architecture](../04-transformer-architecture/)

**What you'll learn:**
- Language modeling objectives: CLM, MLM, prefix LM
- Data pipelines for pretraining: tokenization, packing, padding
- The full pretraining loop with gradient accumulation and mixed precision
- Learning rate schedules: warmup + cosine decay
- Interpreting loss curves and diagnosing training failures
- Checkpointing: saving and resuming training state

**What you'll build:**
- A complete pretraining loop from scratch with all production essentials

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import json
import tempfile
from typing import Optional, List, Tuple

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Language Modeling Objectives

Pretraining is the phase where a language model learns the statistical structure of language from a massive corpus. The choice of training objective fundamentally shapes what the model learns and what it is useful for. There are three major objectives used in modern LLMs, each with distinct strengths.

**Causal Language Modeling (CLM)** is the autoregressive objective used by GPT, LLaMA, and most modern LLMs. The model predicts the next token given all preceding tokens. A causal attention mask ensures the model can only attend to past positions. The loss is the cross-entropy between predicted and actual next tokens, averaged over all positions. CLM models excel at generation because they are trained to produce text left-to-right.

**Masked Language Modeling (MLM)** is the objective introduced by BERT. A fraction of tokens (typically 15%) are replaced with a [MASK] token, and the model predicts the original tokens. MLM allows the model to use bidirectional context — both left and right of the masked position — which gives it an advantage for understanding tasks like classification and NER. However, MLM models are not natural generators because they are not trained autoregressively.

**Prefix Language Modeling** is a hybrid approach used by T5 and UL2. Part of the input serves as a bidirectional prefix (the model can attend to all prefix tokens from any prefix position), and the remainder is generated autoregressively. This combines the bidirectional understanding of MLM with the generation capability of CLM. The boundary between prefix and generation is typically set randomly during training.

In [ ]:
# A minimal transformer we'll use throughout this notebook
class MiniTransformerLM(nn.Module):
    """Small transformer for demonstrating pretraining objectives."""
    def __init__(self, vocab_size=5000, d_model=128, n_heads=4, n_layers=2, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(positions)
        if attention_mask is not None:
            x = self.transformer(x, mask=attention_mask, is_causal=False)
        else:
            causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
            x = self.transformer(x, mask=causal_mask, is_causal=True)
        x = self.ln_f(x)
        return self.lm_head(x)

model = MiniTransformerLM().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
def clm_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    Causal Language Modeling loss.
    logits: (B, T, V) — model output
    labels: (B, T) — target token ids (shifted by 1 relative to input)
    
    We predict position t+1 from position t, so we shift:
    - logits[:, :-1, :] predicts positions 1..T-1
    - labels[:, 1:] are the targets at positions 1..T-1
    """
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )
    return loss

# Demonstrate CLM loss
batch = torch.randint(0, 5000, (4, 64), device=device)
logits = model(batch)
loss = clm_loss(logits, batch)
print(f"CLM loss: {loss.item():.4f}")
print(f"Expected random loss: {math.log(5000):.4f} (ln(vocab_size))")

In [ ]:
def mlm_loss(
    model: nn.Module,
    input_ids: torch.Tensor,
    mask_token_id: int = 1,
    mask_prob: float = 0.15
) -> torch.Tensor:
    """
    Masked Language Modeling loss.
    Randomly masks 15% of tokens, and computes loss only on masked positions.
    Following BERT: 80% [MASK], 10% random, 10% unchanged.
    """
    labels = input_ids.clone()
    masked_input = input_ids.clone()

    # Create mask: which positions to predict
    probability_matrix = torch.full(input_ids.shape, mask_prob, device=input_ids.device)
    masked_indices = torch.bernoulli(probability_matrix).bool()

    # Set labels to -100 for non-masked positions (ignored by cross_entropy)
    labels[~masked_indices] = -100

    # 80% of the time, replace with [MASK]
    indices_replaced = torch.bernoulli(torch.full(input_ids.shape, 0.8, device=input_ids.device)).bool() & masked_indices
    masked_input[indices_replaced] = mask_token_id

    # 10% of the time, replace with random token
    indices_random = torch.bernoulli(torch.full(input_ids.shape, 0.5, device=input_ids.device)).bool() & masked_indices & ~indices_replaced
    random_tokens = torch.randint(2, model.vocab_size, input_ids.shape, device=input_ids.device)
    masked_input[indices_random] = random_tokens[indices_random]

    # 10% of the time, keep original (already the case)

    # Forward with no causal mask (bidirectional attention)
    logits = model(masked_input, attention_mask=torch.zeros(input_ids.size(1), input_ids.size(1), device=input_ids.device))
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
    return loss

loss = mlm_loss(model, batch)
print(f"MLM loss: {loss.item():.4f}")

In [ ]:
def prefix_lm_loss(
    model: nn.Module,
    input_ids: torch.Tensor,
    prefix_fraction: float = 0.5
) -> torch.Tensor:
    """
    Prefix LM loss.
    The prefix portion uses bidirectional attention.
    The generation portion uses causal attention.
    Loss is only computed on the generation portion.
    """
    B, T = input_ids.shape
    prefix_len = int(T * prefix_fraction)

    # Build the attention mask:
    # - prefix tokens can attend to all prefix tokens (bidirectional)
    # - generation tokens can attend to all prefix tokens + causal within generation
    attn_mask = torch.zeros(T, T, device=input_ids.device)
    # Generation portion: causal mask
    for i in range(prefix_len, T):
        attn_mask[i, i+1:] = float('-inf')  # can't attend to future
    # All positions can attend to prefix (already 0 = allowed)

    logits = model(input_ids, attention_mask=attn_mask)

    # Loss only on generation portion (after prefix)
    gen_logits = logits[:, prefix_len:-1, :]
    gen_labels = input_ids[:, prefix_len+1:]
    loss = F.cross_entropy(
        gen_logits.contiguous().view(-1, gen_logits.size(-1)),
        gen_labels.contiguous().view(-1)
    )
    return loss

loss = prefix_lm_loss(model, batch)
print(f"Prefix LM loss: {loss.item():.4f}")

### Comparing the Three Objectives

| Property | CLM | MLM | Prefix LM |
|----------|-----|-----|----------|
| Attention | Causal (left-to-right) | Bidirectional | Prefix: bidirectional, Suffix: causal |
| Loss computed on | All tokens | ~15% masked tokens | Suffix tokens only |
| Token efficiency | High (every token is a training signal) | Low (only 15% of tokens) | Medium |
| Generation quality | Excellent | Poor (not autoregressive) | Good |
| Understanding tasks | Good | Excellent | Excellent |
| Examples | GPT, LLaMA, Mistral | BERT, RoBERTa | T5, UL2 |

## 2. Data Pipelines for Pretraining

The data pipeline for pretraining is deceptively complex and has an enormous impact on model quality. A pretraining corpus typically consists of trillions of tokens from diverse sources — web crawls, books, code repositories, scientific papers, and more. The pipeline must tokenize this data, organize it into fixed-length sequences, and feed it to the model efficiently.

**Tokenization** converts raw text into integer token IDs. Modern LLMs use Byte-Pair Encoding (BPE) or SentencePiece, producing vocabularies of 32K-128K tokens. The tokenizer is trained on the corpus before the model, and its quality matters: a good tokenizer compresses text efficiently (more information per token) and handles rare words gracefully.

**Packing** is the strategy of concatenating multiple documents into a single training sequence, separated by a special end-of-document token. This avoids wasting compute on padding tokens. If a document is shorter than the sequence length, we pack the next document into the remaining space. If a document is longer, we split it across multiple sequences. Packing significantly improves training efficiency — without it, short documents waste most of the sequence length on padding.

**Padding** is the simpler alternative: each document occupies its own sequence, padded to the maximum length. This preserves document boundaries cleanly but wastes substantial compute. In practice, pretraining always uses packing, while fine-tuning sometimes uses padding for simplicity.

In [ ]:
def pack_sequences(
    tokenized_documents: List[List[int]],
    seq_length: int,
    eos_token_id: int = 0
) -> List[List[int]]:
    """
    Pack tokenized documents into fixed-length sequences.
    Documents are concatenated with EOS tokens between them.
    
    Args:
        tokenized_documents: List of token ID lists, one per document.
        seq_length: Fixed sequence length for training.
        eos_token_id: End-of-sequence token to insert between documents.
    
    Returns:
        List of packed sequences, each of length seq_length.
    """
    # Concatenate all documents with EOS separators
    all_tokens = []
    for doc in tokenized_documents:
        all_tokens.extend(doc)
        all_tokens.append(eos_token_id)

    # Split into fixed-length chunks
    packed = []
    for i in range(0, len(all_tokens) - seq_length + 1, seq_length):
        packed.append(all_tokens[i:i + seq_length])

    return packed


def pad_sequences(
    tokenized_documents: List[List[int]],
    seq_length: int,
    pad_token_id: int = 0
) -> Tuple[List[List[int]], List[List[int]]]:
    """
    Pad each document to seq_length. Truncate if longer.
    Returns (padded_sequences, attention_masks).
    """
    padded = []
    masks = []
    for doc in tokenized_documents:
        if len(doc) >= seq_length:
            padded.append(doc[:seq_length])
            masks.append([1] * seq_length)
        else:
            pad_len = seq_length - len(doc)
            padded.append(doc + [pad_token_id] * pad_len)
            masks.append([1] * len(doc) + [0] * pad_len)
    return padded, masks


# Simulate some tokenized documents of varying lengths
np.random.seed(42)
documents = [np.random.randint(2, 5000, size=np.random.randint(10, 200)).tolist() for _ in range(100)]
doc_lengths = [len(d) for d in documents]
print(f"Number of documents: {len(documents)}")
print(f"Document lengths: min={min(doc_lengths)}, max={max(doc_lengths)}, mean={np.mean(doc_lengths):.0f}")

seq_length = 128
packed = pack_sequences(documents, seq_length)
padded, masks = pad_sequences(documents, seq_length)

packed_tokens = len(packed) * seq_length
padded_tokens = len(padded) * seq_length
padded_real = sum(sum(m) for m in masks)

print(f"\nPacking: {len(packed)} sequences, {packed_tokens} total tokens (all useful)")
print(f"Padding: {len(padded)} sequences, {padded_tokens} total tokens ({padded_real} real, {padded_tokens - padded_real} padding)")
print(f"Packing efficiency: {packed_tokens / padded_tokens * 100:.1f}% of padding token count")
print(f"Padding waste: {(padded_tokens - padded_real) / padded_tokens * 100:.1f}% wasted compute")

In [ ]:
# Visualize packing vs padding efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Document length distribution
axes[0].hist(doc_lengths, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(seq_length, color='red', linestyle='--', label=f'seq_length={seq_length}')
axes[0].set_xlabel('Document Length (tokens)')
axes[0].set_ylabel('Count')
axes[0].set_title('Document Length Distribution')
axes[0].legend()

# Efficiency comparison
categories = ['Packing', 'Padding']
useful = [packed_tokens, padded_real]
waste = [0, padded_tokens - padded_real]
axes[1].bar(categories, useful, label='Useful tokens', color='steelblue')
axes[1].bar(categories, waste, bottom=useful, label='Wasted (padding)', color='lightcoral')
axes[1].set_ylabel('Total Tokens')
axes[1].set_title('Token Efficiency: Packing vs Padding')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
class PackedPretrainingDataset(Dataset):
    """Dataset that packs tokenized documents into fixed-length sequences."""

    def __init__(self, documents: List[List[int]], seq_length: int, eos_token_id: int = 0):
        self.seq_length = seq_length
        self.sequences = pack_sequences(documents, seq_length, eos_token_id)
        print(f"Created dataset with {len(self.sequences)} packed sequences of length {seq_length}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        tokens = torch.tensor(self.sequences[idx], dtype=torch.long)
        # For CLM: input is tokens[:-1], target is tokens[1:]
        # But we return full sequence and let the loss function handle shifting
        return tokens

dataset = PackedPretrainingDataset(documents, seq_length=128)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
batch = next(iter(loader))
print(f"Batch shape: {batch.shape}")

## 3. The Pretraining Loop

A production pretraining loop involves several components beyond a naive `loss.backward(); optimizer.step()`. The two most important additions are **gradient accumulation** and **mixed precision training**.

**Gradient accumulation** allows you to simulate a large batch size without needing enough GPU memory to hold the full batch. Instead of updating the model after every micro-batch, you accumulate gradients over multiple micro-batches and then perform a single optimizer step. If you accumulate over `K` micro-batches of size `B`, the effective batch size is `K * B`. The gradients must be divided by `K` (or equivalently, the loss must be divided by `K` before each backward pass) to maintain the correct scale. Large effective batch sizes (in the thousands of sequences) are critical for stable pretraining.

**Mixed precision training** uses lower-precision floating-point (float16 or bfloat16) for most computations while keeping a master copy of weights in float32. This roughly halves memory usage and doubles throughput on modern GPUs. PyTorch's `torch.cuda.amp` provides automatic mixed precision: the forward pass runs in float16/bfloat16, while the optimizer step operates on float32 master weights. A `GradScaler` is used with float16 to prevent underflow in gradients, though bfloat16 has sufficient dynamic range that scaling is usually unnecessary.

In [ ]:
def pretrain_loop(
    model: nn.Module,
    dataloader: DataLoader,
    num_epochs: int = 2,
    lr: float = 3e-4,
    weight_decay: float = 0.1,
    grad_accum_steps: int = 4,
    use_amp: bool = True,
    max_grad_norm: float = 1.0,
    log_interval: int = 5
) -> dict:
    """
    Full pretraining loop with gradient accumulation and mixed precision.
    
    Returns dict with training metrics.
    """
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.95)
    )

    # Use bfloat16 if available (no scaler needed), else float16 with scaler
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and not use_bf16 and torch.cuda.is_available()))
    autocast_device = 'cuda' if torch.cuda.is_available() else 'cpu'

    metrics = {'loss': [], 'grad_norm': [], 'step': []}
    global_step = 0
    accumulated_loss = 0.0

    model.train()
    for epoch in range(num_epochs):
        for micro_step, batch in enumerate(dataloader):
            batch = batch.to(device)

            # Mixed precision forward pass
            with torch.autocast(device_type=autocast_device, dtype=amp_dtype, enabled=use_amp):
                logits = model(batch)
                loss = clm_loss(logits, batch)
                # Scale loss by accumulation steps
                loss = loss / grad_accum_steps

            # Backward pass (scaled if using float16)
            scaler.scale(loss).backward()
            accumulated_loss += loss.item()

            # Update weights every grad_accum_steps micro-batches
            if (micro_step + 1) % grad_accum_steps == 0:
                # Unscale gradients for clipping
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1
                metrics['loss'].append(accumulated_loss)
                metrics['grad_norm'].append(grad_norm.item() if isinstance(grad_norm, torch.Tensor) else grad_norm)
                metrics['step'].append(global_step)

                if global_step % log_interval == 0:
                    print(f"  Step {global_step:4d} | Loss: {accumulated_loss:.4f} | Grad norm: {metrics['grad_norm'][-1]:.4f}")

                accumulated_loss = 0.0

        print(f"Epoch {epoch+1}/{num_epochs} complete")

    return metrics

# Reset model and train
model = MiniTransformerLM().to(device)
metrics = pretrain_loop(model, loader, num_epochs=3, grad_accum_steps=2, use_amp=False, log_interval=2)
print(f"\nTotal steps: {len(metrics['loss'])}")

In [ ]:
# Visualize training metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(metrics['step'], metrics['loss'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(metrics['step'], metrics['grad_norm'], color='darkorange', alpha=0.7)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms During Training')

plt.tight_layout()
plt.show()

## 4. Learning Rate Schedules

The learning rate schedule is one of the most important hyperparameters in pretraining. Nearly all modern LLMs use a **linear warmup followed by cosine decay**, first popularized by the GPT-2 training recipe.

**Linear warmup** gradually increases the learning rate from 0 to the peak value over a fixed number of steps (typically 0.1-2% of total training steps). Warmup prevents the model from making large, destabilizing updates in the early phase when the loss landscape is most chaotic and the Adam optimizer's second-moment estimates have not yet converged. Without warmup, the initial gradient updates can be disproportionately large, leading to loss spikes or even divergence.

**Cosine decay** smoothly reduces the learning rate from its peak to a small final value (often 10% of the peak or even 0) following a cosine curve. This schedule is preferred over linear decay because it spends more time at higher learning rates (where the model learns most) and then gently anneals. The cosine shape also provides a natural "cool-down" period that helps the model converge to a flatter minimum, which typically generalizes better.

An important practical consideration is the **total number of training steps**, which must be decided in advance for cosine decay (unlike linear decay). If training is extended beyond the planned steps, the learning rate will already be near its minimum, which can waste compute. Some practitioners use a warmup-stable-decay schedule (WSD) as an alternative that is more flexible with respect to total training duration.

In [ ]:
def get_cosine_schedule_with_warmup(
    optimizer: torch.optim.Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    min_lr_ratio: float = 0.1
) -> torch.optim.lr_scheduler.LambdaLR:
    """
    Linear warmup followed by cosine decay to min_lr_ratio * peak_lr.
    """
    def lr_lambda(current_step: int) -> float:
        # Warmup phase
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        # Cosine decay phase
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        # Decay from 1.0 to min_lr_ratio
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Visualize different schedule configurations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

total_steps = 10000
peak_lr = 3e-4

# Left: varying warmup fraction
for warmup_frac in [0.01, 0.02, 0.05, 0.10]:
    warmup_steps = int(total_steps * warmup_frac)
    dummy_opt = torch.optim.SGD([torch.nn.Parameter(torch.zeros(1))], lr=peak_lr)
    scheduler = get_cosine_schedule_with_warmup(dummy_opt, warmup_steps, total_steps)
    lrs = []
    for step in range(total_steps):
        lrs.append(dummy_opt.param_groups[0]['lr'])
        dummy_opt.step()
        scheduler.step()
    axes[0].plot(lrs, label=f'warmup={warmup_frac:.0%}', alpha=0.8)

axes[0].set_xlabel('Step')
axes[0].set_ylabel('Learning Rate')
axes[0].set_title('Effect of Warmup Fraction')
axes[0].legend()

# Right: varying min LR ratio
for min_ratio in [0.0, 0.05, 0.1, 0.2]:
    warmup_steps = int(total_steps * 0.02)
    dummy_opt = torch.optim.SGD([torch.nn.Parameter(torch.zeros(1))], lr=peak_lr)
    scheduler = get_cosine_schedule_with_warmup(dummy_opt, warmup_steps, total_steps, min_lr_ratio=min_ratio)
    lrs = []
    for step in range(total_steps):
        lrs.append(dummy_opt.param_groups[0]['lr'])
        dummy_opt.step()
        scheduler.step()
    axes[1].plot(lrs, label=f'min_ratio={min_ratio}', alpha=0.8)

axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Effect of Minimum LR Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Warmup-Stable-Decay (WSD) schedule — a popular alternative
def get_wsd_schedule(
    optimizer: torch.optim.Optimizer,
    num_warmup_steps: int,
    num_stable_steps: int,
    num_decay_steps: int,
    min_lr_ratio: float = 0.0
) -> torch.optim.lr_scheduler.LambdaLR:
    """
    Warmup-Stable-Decay schedule.
    Advantage: you don't need to know total steps in advance.
    """
    def lr_lambda(current_step: int) -> float:
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        elif current_step < num_warmup_steps + num_stable_steps:
            return 1.0
        else:
            decay_step = current_step - num_warmup_steps - num_stable_steps
            progress = min(1.0, float(decay_step) / float(max(1, num_decay_steps)))
            cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
            return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Compare cosine vs WSD
fig, ax = plt.subplots(figsize=(10, 5))
total = 10000

for name, sched_fn in [
    ('Cosine w/ warmup', lambda opt: get_cosine_schedule_with_warmup(opt, 200, total)),
    ('WSD', lambda opt: get_wsd_schedule(opt, 200, 7000, 2800)),
]:
    opt = torch.optim.SGD([torch.nn.Parameter(torch.zeros(1))], lr=peak_lr)
    sched = sched_fn(opt)
    lrs = []
    for s in range(total):
        lrs.append(opt.param_groups[0]['lr'])
        opt.step(); sched.step()
    ax.plot(lrs, label=name, alpha=0.8)

ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine vs Warmup-Stable-Decay')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Loss Curves — Diagnosing Training Problems

Reading loss curves is one of the most important practical skills for anyone training language models. A healthy loss curve decreases smoothly with occasional minor fluctuations. When things go wrong, the loss curve tells you exactly what happened, and an experienced practitioner can diagnose the problem at a glance.

**Loss spikes** are sudden, sharp increases in loss that recover within a few hundred steps. They are the most common training failure mode and are typically caused by a batch of unusual data (e.g., very long documents, rare languages, or corrupted data) or by the learning rate being too high. Mild spikes are normal and harmless; severe spikes can permanently damage model quality. Gradient clipping mitigates their severity.

**Divergence** is when the loss increases and never recovers, eventually reaching NaN. The most common cause is a learning rate that is too high. Other causes include numerical instability (especially in float16 without proper scaling), bugs in the data pipeline (e.g., all-padding batches), or incorrect loss computation. The fix is usually to lower the learning rate, add gradient clipping, or switch to bfloat16.

**Plateaus** occur when the loss stops decreasing for an extended period. This can indicate that the learning rate has decayed too much too early, that the model has saturated its capacity for the available data, or that there is a data quality issue (e.g., too much duplicate data). Plateaus are sometimes followed by sudden improvements ("grokking"), but more often they indicate a problem.

**Oscillation** manifests as a loss that bounces up and down without a clear downward trend. This typically indicates that the batch size is too small or that there is high variance in the data distribution across batches.

In [ ]:
def simulate_loss_curve(mode: str, num_steps: int = 5000) -> np.ndarray:
    """Simulate different loss curve pathologies."""
    steps = np.arange(num_steps)
    noise = np.random.randn(num_steps) * 0.02

    if mode == 'healthy':
        # Smooth power-law decay
        loss = 8.5 * (steps + 1) ** (-0.15) + noise

    elif mode == 'spikes':
        loss = 8.5 * (steps + 1) ** (-0.15) + noise
        # Add spikes at random positions
        for spike_pos in [800, 2200, 3500]:
            spike = np.zeros(num_steps)
            spike_len = 50
            spike[spike_pos:spike_pos+spike_len] = np.exp(-np.arange(spike_len) / 10) * 2.0
            loss += spike

    elif mode == 'divergence':
        loss = 8.5 * (steps + 1) ** (-0.15) + noise
        # Diverge at step 2000
        diverge_start = 2000
        loss[diverge_start:] = loss[diverge_start] + np.cumsum(
            np.abs(np.random.randn(num_steps - diverge_start)) * 0.01
        )

    elif mode == 'plateau':
        loss = 8.5 * (steps + 1) ** (-0.15) + noise
        # Plateau between 1500-3500
        plateau_val = loss[1500]
        loss[1500:3500] = plateau_val + np.random.randn(2000) * 0.02
        loss[3500:] = plateau_val * (steps[3500:] - 3500 + 1) ** (-0.05) + noise[3500:]

    elif mode == 'oscillation':
        base = 8.5 * (steps + 1) ** (-0.15)
        oscillation = 0.3 * np.sin(steps / 20) * np.exp(-steps / 8000)
        loss = base + oscillation + noise * 3

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return loss


fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

modes = ['healthy', 'spikes', 'divergence', 'plateau', 'oscillation']
titles = [
    'Healthy Training',
    'Loss Spikes (data/LR issues)',
    'Divergence (LR too high)',
    'Plateau (capacity/data)',
    'Oscillation (small batch size)'
]
colors = ['forestgreen', 'darkorange', 'red', 'purple', 'steelblue']

for i, (mode, title, color) in enumerate(zip(modes, titles, colors)):
    loss = simulate_loss_curve(mode)
    axes[i].plot(loss, color=color, alpha=0.7, linewidth=0.8)
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel('Step')
    axes[i].set_ylabel('Loss')
    axes[i].set_ylim(bottom=0)

axes[5].axis('off')
axes[5].text(0.1, 0.8, 'Diagnosis Guide:', fontweight='bold', fontsize=12, transform=axes[5].transAxes)
axes[5].text(0.1, 0.65, '- Spikes: clip gradients, check data', fontsize=10, transform=axes[5].transAxes)
axes[5].text(0.1, 0.50, '- Divergence: lower LR, use bf16', fontsize=10, transform=axes[5].transAxes)
axes[5].text(0.1, 0.35, '- Plateau: check LR schedule, data', fontsize=10, transform=axes[5].transAxes)
axes[5].text(0.1, 0.20, '- Oscillation: increase batch size', fontsize=10, transform=axes[5].transAxes)

plt.suptitle('Loss Curve Pathologies in Pretraining', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Checkpointing — Save, Load, and Resume Training

Pretraining runs last days to months on expensive GPU clusters, so checkpointing is not optional — it is critical infrastructure. A checkpoint must save everything needed to resume training exactly where it left off, which includes the model weights, optimizer state (the Adam momentum buffers), learning rate scheduler state, the current training step, and ideally the random number generator states for reproducibility.

A common mistake is saving only the model weights. This is fine for inference but useless for resuming training, because the Adam optimizer maintains per-parameter momentum estimates that take thousands of steps to re-warm. Resuming with fresh optimizer state will cause a temporary spike in loss and waste compute.

Checkpointing frequency involves a tradeoff between safety and I/O overhead. Saving too frequently wastes time on disk I/O and storage; saving too infrequently risks losing days of compute if a node fails. A common strategy is to save every 1000-5000 steps, keeping only the last N checkpoints to limit disk usage. Some teams also save a checkpoint whenever a new minimum validation loss is achieved.

In [ ]:
def save_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    step: int,
    loss: float,
    config: dict = None
):
    """Save a complete training checkpoint."""
    checkpoint = {
        'step': step,
        'loss': loss,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'torch_rng_state': torch.get_rng_state(),
        'numpy_rng_state': np.random.get_state(),
    }
    if config:
        checkpoint['config'] = config
    if torch.cuda.is_available():
        checkpoint['cuda_rng_state'] = torch.cuda.get_rng_state()
    torch.save(checkpoint, path)
    print(f"Checkpoint saved to {path} at step {step}")


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer = None,
    scheduler: torch.optim.lr_scheduler._LRScheduler = None
) -> dict:
    """Load a training checkpoint, restoring all state."""
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'torch_rng_state' in checkpoint:
        torch.set_rng_state(checkpoint['torch_rng_state'])
    if 'numpy_rng_state' in checkpoint:
        np.random.set_state(checkpoint['numpy_rng_state'])
    if torch.cuda.is_available() and 'cuda_rng_state' in checkpoint:
        torch.cuda.set_rng_state(checkpoint['cuda_rng_state'])
    print(f"Checkpoint loaded from {path} at step {checkpoint['step']}")
    return checkpoint

print("Checkpoint utilities defined.")

In [ ]:
def pretrain_with_checkpointing(
    model: nn.Module,
    dataloader: DataLoader,
    num_epochs: int = 3,
    lr: float = 3e-4,
    weight_decay: float = 0.1,
    grad_accum_steps: int = 2,
    max_grad_norm: float = 1.0,
    checkpoint_dir: str = '/tmp/checkpoints',
    checkpoint_every: int = 10,
    resume_from: str = None
) -> dict:
    """
    Full pretraining loop with LR schedule, gradient accumulation, and checkpointing.
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    total_steps = len(dataloader) // grad_accum_steps * num_epochs
    warmup_steps = max(1, int(total_steps * 0.05))

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.95))
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    start_step = 0
    if resume_from and os.path.exists(resume_from):
        ckpt = load_checkpoint(resume_from, model, optimizer, scheduler)
        start_step = ckpt['step']
        print(f"Resuming from step {start_step}")

    metrics = {'loss': [], 'lr': [], 'step': []}
    global_step = start_step
    accumulated_loss = 0.0

    model.train()
    for epoch in range(num_epochs):
        for micro_step, batch in enumerate(dataloader):
            batch = batch.to(device)
            logits = model(batch)
            loss = clm_loss(logits, batch) / grad_accum_steps
            loss.backward()
            accumulated_loss += loss.item()

            if (micro_step + 1) % grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1
                current_lr = optimizer.param_groups[0]['lr']
                metrics['loss'].append(accumulated_loss)
                metrics['lr'].append(current_lr)
                metrics['step'].append(global_step)
                accumulated_loss = 0.0

                # Save checkpoint
                if global_step % checkpoint_every == 0:
                    ckpt_path = os.path.join(checkpoint_dir, f'ckpt_step_{global_step}.pt')
                    save_checkpoint(ckpt_path, model, optimizer, scheduler, global_step, metrics['loss'][-1])

    return metrics

# Train with checkpointing
model = MiniTransformerLM().to(device)
ckpt_dir = tempfile.mkdtemp()
metrics = pretrain_with_checkpointing(model, loader, num_epochs=3, checkpoint_dir=ckpt_dir, checkpoint_every=10)
print(f"\nTraining complete. Steps: {len(metrics['loss'])}")

In [ ]:
# Show loss and LR side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(metrics['step'], metrics['loss'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss with Cosine Schedule')

axes[1].plot(metrics['step'], metrics['lr'], color='darkorange', alpha=0.8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule (warmup + cosine)')

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate checkpoint resume: load the last checkpoint and verify state
import glob

checkpoints = sorted(glob.glob(os.path.join(ckpt_dir, 'ckpt_step_*.pt')))
print(f"Available checkpoints: {[os.path.basename(c) for c in checkpoints]}")

if checkpoints:
    # Create a fresh model and load checkpoint
    fresh_model = MiniTransformerLM().to(device)

    # Generate some text with the fresh (random) model
    test_input = torch.randint(0, 5000, (1, 16), device=device)
    with torch.no_grad():
        fresh_logits = fresh_model(test_input)

    # Load checkpoint
    ckpt = load_checkpoint(checkpoints[-1], fresh_model)
    with torch.no_grad():
        loaded_logits = fresh_model(test_input)

    # Compare with the trained model
    with torch.no_grad():
        trained_logits = model(test_input)

    print(f"\nLogit comparison (should be near 0 for loaded vs trained):")
    print(f"  Fresh vs Trained: {(fresh_logits - trained_logits).abs().mean().item():.6f}")
    print(f"  Loaded vs Trained: {(loaded_logits - trained_logits).abs().mean().item():.6f}")

In [ ]:
# Demonstrate resume training from checkpoint
if checkpoints:
    mid_ckpt = checkpoints[len(checkpoints) // 2]  # Resume from middle checkpoint
    print(f"Resuming from: {os.path.basename(mid_ckpt)}")

    resumed_model = MiniTransformerLM().to(device)
    resumed_metrics = pretrain_with_checkpointing(
        resumed_model, loader,
        num_epochs=1,
        checkpoint_dir=ckpt_dir,
        checkpoint_every=100,  # Don't save during this short run
        resume_from=mid_ckpt
    )
    print(f"Resumed training for {len(resumed_metrics['loss'])} additional steps")

## Summary

In this notebook we covered the foundational components of pretraining a language model:

1. **Language modeling objectives** — CLM (autoregressive), MLM (bidirectional), and prefix LM (hybrid). CLM dominates modern LLMs because it naturally produces generators.

2. **Data pipelines** — Packing eliminates wasted compute from padding, making it essential for pretraining. Good tokenization is the first link in the quality chain.

3. **Training loop** — Gradient accumulation simulates large batches; mixed precision halves memory and doubles speed.

4. **Learning rate schedules** — Linear warmup + cosine decay is the standard. WSD is a flexible alternative when you do not know total steps in advance.

5. **Loss curves** — Learn to read them: spikes, divergence, plateaus, and oscillation each have distinct causes and fixes.

6. **Checkpointing** — Save model + optimizer + scheduler + RNG state. Without this, multi-day training runs are impossible.

**Next:** [05-pretraining/advanced.ipynb](./advanced.ipynb) covers distributed training with DDP, DeepSpeed, and FSDP.